# 03 · Historical Launch Table Web Scraping
**A separate data-source exercise — not a replacement for the IBM API-derived dataset**

The original notebook parsed a historical Wikipedia revision of the *List of Falcon 9 and Falcon Heavy launches*. This modern version preserves the **archived page revision**, saves its HTML for repeatability, extracts launch-like table rows, and audits the extraction. Wikipedia HTML is not a stable API: parsing failures are reported rather than hidden.

**Important:** These Wikipedia records are a separate source and are **not merged** into the 90-row IBM analytical dataset. The output is an independently sourced supplementary table. Verify extracted rows against the page before using them for analysis. Original IBM teaching notebook remains in `original-coursework/`.

## 1. Configuration

In [1]:
from pathlib import Path
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
ARCHIVE_URL = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
HTML_FILE = RAW / 'wikipedia_falcon_launches_oldid_1027686922.html'
OUTPUT = PROCESSED / 'wikipedia_falcon_launches_archived.csv'
REFRESH = False

## 2. Download once, then work from the cached HTML

In [2]:
if HTML_FILE.exists() and not REFRESH:
    html = HTML_FILE.read_text(encoding='utf-8')
    print('Loaded archived HTML cache')
else:
    response = requests.get(ARCHIVE_URL, headers={'User-Agent':'EducationalSpaceXPortfolio/1.0 (research notebook)'}, timeout=30)
    response.raise_for_status()
    html = response.text
    HTML_FILE.write_text(html, encoding='utf-8')
    print('Saved archived HTML')
soup = BeautifulSoup(html, 'html.parser')
print('Page title:', soup.title.get_text(' ', strip=True) if soup.title else 'missing')
assert soup.find('table'), 'No HTML tables found; inspect the saved page'

Saved archived HTML
Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


## 3. Extract tabular records conservatively

This parser reads table headings and row cells rather than assuming the third HTML table is always the launch table. It keeps the source table number and original text so suspicious records can be traced back. It does **not** claim to reconstruct every payload/customer/landing field perfectly.

In [3]:
records=[]
for table_idx, table in enumerate(soup.select('table.wikitable')):
    headers = [h.get_text(' ', strip=True) for h in table.select('tr:first-child th')]
    header_text = ' '.join(headers).lower()
    if not ('flight' in header_text or 'launch' in header_text or 'date' in header_text):
        continue
    for row in table.select('tr'):
        cells = row.find_all(['th','td'], recursive=False)
        if len(cells) < 4: continue
        first = cells[0].get_text(' ', strip=True)
        flight_match = re.match(r'^\s*(\d{1,4})\b', first)
        if not flight_match: continue
        records.append({
            'SourceTable': table_idx,
            'FlightNumber': int(flight_match.group(1)),
            'FirstCell': first,
            'CellCount': len(cells),
            'RowText': ' | '.join(c.get_text(' ', strip=True) for c in cells),
        })
scraped = pd.DataFrame.from_records(records)
if scraped.empty:
    raise ValueError('No flight-number rows extracted. Inspect the cached HTML and update selectors.')
scraped = scraped.drop_duplicates(subset=['SourceTable','FlightNumber','RowText']).reset_index(drop=True)
print('Extracted launch-like rows:',len(scraped))
display(scraped.head(10))

Extracted launch-like rows: 156


,SourceTable,FlightNumber,FirstCell,CellCount,RowText
0,0,1,1,10,"1 | 4 June 2010, 18:45 | F9 v1.0 [ 7 ] B0003.1..."
1,0,2,2,10,"2 | 8 December 2010, 15:43 [ 13 ] | F9 v1.0 [ ..."
2,0,3,3,10,"3 | 22 May 2012, 07:44 [ 17 ] | F9 v1.0 [ 7 ] ..."
3,0,4,4,10,"4 | 8 October 2012, 00:35 [ 21 ] | F9 v1.0 [ 7..."
4,0,5,5,10,"5 | 1 March 2013, 15:10 | F9 v1.0 [ 7 ] B0007...."
5,0,6,6,10,"6 | 29 September 2013, 16:00 [ 30 ] | F9 v1.1 ..."
6,0,7,7,10,"7 | 3 December 2013, 22:41 [ 34 ] | F9 v1.1 B1..."
7,1,8,8,10,"8 | 6 January 2014, 22:06 [ 41 ] | F9 v1.1 | C..."
8,1,9,9,10,"9 | 18 April 2014, 19:25 [ 21 ] | F9 v1.1 | Ca..."
9,1,10,10,10,"10 | 14 July 2014, 15:15 | F9 v1.1 | Cape Cana..."


## 4. Audit and save

In [4]:
print('Rows per source table:')
display(scraped.groupby('SourceTable').size().rename('rows').to_frame())
print('Repeated flight numbers across tables:', int(scraped['FlightNumber'].duplicated().sum()))
print('These counts are diagnostic, not evidence of unique missions.')
scraped.to_csv(OUTPUT, index=False)
print('Saved:', OUTPUT)

Rows per source table:


,rows
SourceTable,
0,7
1,6
2,7
3,8
4,18
5,20
6,11
7,26
8,18


Repeated flight numbers across tables: 31
These counts are diagnostic, not evidence of unique missions.
Saved: c:\Users\kanwa\Downloads\New folder\notebooks\data\processed\wikipedia_falcon_launches_archived.csv


## Limitations
- The archived HTML page may have nested tables, rowspan/colspan cells, footnotes and multiple launches per table. We intentionally retain **raw row text** rather than inventing structured mission fields.
- **Manual spot checks are required** before treating this export as an authoritative flight dataset.
- This supplementary exercise is independent of the IBM 90-row data pipeline. For production extraction, use an explicit, tested table schema and archival fixture tests.